# 2. Data visualization

Explore the cleaned loan-default dataset with distribution charts, default rates by borrower group, and a numeric-feature correlation heatmap. Run `01_Data_Cleaning.ipynb` first to create the processed CSV.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "loan_default_cleaned.csv"
if not CLEAN_PATH.exists():
    raise FileNotFoundError("Run notebooks/01_Data_Cleaning.ipynb first to create the cleaned dataset.")

df = pd.read_csv(CLEAN_PATH)
TARGET = "Default"
sns.set_theme(style="whitegrid", context="notebook")
print(f"Loaded {len(df):,} cleaned loans; observed default rate: {df[TARGET].mean():.2%}")

## Outcome balance and feature distributions

The dataset does not specify a currency, so monetary values are shown as numeric amounts without a currency label.

In [ ]:
sample = df.sample(n=min(10_000, len(df)), random_state=42)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, x=TARGET, ax=axes[0, 0], color="#38BDF8")
axes[0, 0].set(title="Loan outcome counts", xlabel="Default (1 = yes)", ylabel="Loans")

sns.histplot(data=sample, x="Age", hue=TARGET, bins=25, stat="density", common_norm=False,
             element="step", ax=axes[0, 1])
axes[0, 1].set(title="Age distribution by outcome", ylabel="Within-outcome density")

sns.histplot(data=sample, x="LoanAmount", hue=TARGET, bins=35, stat="density", common_norm=False,
             element="step", ax=axes[1, 0])
axes[1, 0].set(title="Loan amount distribution by outcome", ylabel="Within-outcome density")

sns.histplot(data=sample, x="CreditScore", hue=TARGET, bins=25, stat="density", common_norm=False,
             element="step", ax=axes[1, 1])
axes[1, 1].set(title="Credit score distribution by outcome", ylabel="Within-outcome density")

plt.tight_layout()
plt.show()

## Default rates by group

Group rates describe associations in this dataset; they do not establish that a group characteristic causes default.

In [ ]:
group_columns = ["EmploymentType", "LoanPurpose", "Education", "HasCoSigner"]
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for column, ax in zip(group_columns, axes.flat):
    rates = df.groupby(column, observed=True)[TARGET].agg(default_rate="mean", loans="size")
    rates = rates.sort_values("default_rate", ascending=False)
    sns.barplot(x=rates.index, y=rates["default_rate"] * 100, ax=ax, color="#2DD4BF")
    ax.axhline(df[TARGET].mean() * 100, color="#FB7185", linestyle="--", label="Overall rate")
    ax.set(title=f"Default rate by {column}", xlabel=column, ylabel="Default rate (%)")
    ax.tick_params(axis="x", rotation=25)
    ax.legend()
plt.tight_layout()
plt.show()

display((df.groupby("EmploymentType", observed=True)[TARGET]
           .agg(default_rate="mean", loans="size")
           .assign(default_rate=lambda rates: (rates["default_rate"] * 100).round(2))
           .sort_values("default_rate", ascending=False)))

## Numeric feature correlations

In [ ]:
numeric = df.select_dtypes(include="number")
plt.figure(figsize=(12, 9))
sns.heatmap(numeric.corr(), cmap="vlag", center=0, annot=True, fmt=".2f", square=True)
plt.title("Correlation between numeric features")
plt.tight_layout()
plt.show()